# 00_02 Notebook Basics

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Learn how Databricks notebooks work: cells, magic commands, widgets, <code>display</code>, and running one notebook from another.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Real pipelines are built from many notebooks. Knowing magic commands and <code>%run</code> lets you mix SQL and Python and reuse code instead of copying it. Widgets are how jobs pass parameters into notebooks, which comes up again in <code>04_databricks</code> and <code>07_workflows</code>.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Cells and magic commands</b><br>
A notebook is a list of cells. Each cell runs in the notebook's default language (Python here), unless its first line is a <b>magic command</b>.<br><br>
The magic command must be the <b>first line</b> of the cell.
</div>

| Magic | Purpose |
|---|---|
| `%python` | Run Python |
| `%sql` | Run SQL |
| `%md` | Markdown text |
| `%run` | Run another notebook in this notebook's context |
| `%pip` | Install a Python library for this notebook |
| `%sh` | Run a shell command on the driver (limited on serverless) |

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
An ETL notebook cleans data in Python, then an analyst runs a <code>%sql</code> cell on the same data to check results. A shared <code>config</code> notebook is pulled in with <code>%run</code>.
</div>

## 1. print vs show vs display

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a small DataFrame and shows it three ways.<br><br>
<b>Why it matters</b><br><code>print</code> shows the object description, <code>show()</code> prints a text table, and <code>display()</code> gives an interactive table with sorting and charts.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A schema-like description (<code>DataFrame[id: bigint, name: string, salary: bigint]</code>), a text table, then an interactive table.
</div>

In [0]:
data = [(1, "Amal", 21000), (2, "Vanathaiyan", 15000)]
df = spark.createDataFrame(data, ["id", "name", "salary"])

print(df)       # prints the object description, not the data
df.show()       # plain text table
display(df)     # interactive Databricks table (sort, chart, download)

## 2. Temp view: bridge between Python and SQL

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Registers the DataFrame as a temporary view named <code>employees</code>.<br><br>
<b>Why it matters</b><br>Once a view exists, you can query the same data with SQL, so you can mix Python and SQL in one notebook. Temp views are covered fully in lesson 27.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output. The view now exists for this session.
</div>

In [0]:
df.createOrReplaceTempView("employees")

## 3. SQL magic command

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs a SQL query on the <code>employees</code> view using <code>%sql</code>.<br><br>
<b>Why it matters</b><br><code>%sql</code> must be the first line of the cell. It lets analysts use SQL on data prepared in Python. The result of the last <code>%sql</code> query is also available in Python as the DataFrame <code>_sqldf</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
One row: Amal with 21000.
</div>

In [0]:
%sql
SELECT name,salary
FROM employees
WHERE salary > 16000

## 4. Widgets (parameters)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a text box at the top of the notebook and filters the DataFrame by its value.<br><br>
<b>Why it matters</b><br>Widgets turn a notebook into a parameterized one. Jobs use the same mechanism to pass values like dates or environment names.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A widget appears at the top. Change it to <code>Vanathaiyan</code>, re-run, and the result changes.
</div>

In [0]:
dbutils.widgets.text("employee_name","Amal","Employee Name")
name = dbutils.widgets.get("employee_name")
display(df.filter(df.name == name))

## 5. Remove widgets

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Deletes all widgets from the notebook.<br><br>
<b>Why it matters</b><br>Keeps the notebook clean and avoids old widget values affecting later runs.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The widget bar at the top disappears.
</div>

In [0]:
dbutils.widgets.removeAll()

## 6. Reusing code with %run

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Loads the notebook <code>00_helper</code> into this notebook, so its functions become available here.<br><br>
<b>Why it matters</b><br><code>%run</code> is how you share helper code and config across notebooks without copying it. It must be alone in its cell, and the path is relative to this notebook.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output from the <code>%run</code> cell, then <code>Hello, Databricks!</code> from the next cell.
</div>

In [0]:
%run ./00_helper

In [0]:
print(greet("Databricks"))

## 7. %run vs dbutils.notebook.run

| | `%run ./other` | `dbutils.notebook.run("./other", 60, {...})` |
|---|---|---|
| Context | Same context: functions and variables are shared | Separate run with its own context |
| Parameters | None (uses the caller's variables) | Passed as a dictionary to the child's widgets |
| Return value | None | A string returned with `dbutils.notebook.exit(value)` |
| Typical use | Shared helpers and config | Simple orchestration, running notebooks in a loop |

Notebook workflows are covered in `07_workflows`.

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>%sql</code> cell runs as Python and errors</b><br>
The magic command must be on the first line of the cell.<br><br>
**⛔ Problem: <code>%run</code> fails or says the notebook is not found**<br>Check the relative path, and keep <code>%run</code> alone in its cell.<br><br>
**⛔ Problem: <code>NameError</code> for a function after <code>%run</code>**<br>The <code>%run</code> cell didn't run (or failed). Run it again before the cells that use its functions.<br><br>
<b>⛔ Problem: widget value doesn't change the result</b><br>Re-run the cells after changing the widget, or set the widget's "On widget change" behavior.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between <code>show()</code> and <code>display()</code>?</b><br>
<code>show()</code> is a Spark method that prints a text table. <code>display()</code> is Databricks-specific and gives an interactive table with sorting and charts.<br><br>

<b>🎤 2. How do you share data between Python and SQL cells?</b><br>
Register a temp view with <code>createOrReplaceTempView</code> in Python, then query it in a <code>%sql</code> cell.<br><br>

<b>🎤 3. What is the difference between <code>%run</code> and <code>dbutils.notebook.run</code>?</b><br>
<code>%run</code> includes the other notebook's code and variables in the current notebook. <code>dbutils.notebook.run</code> starts it as a separate run with its own context and returns a value.<br><br>

<b>🎤 4. What are widgets used for?</b><br>
Parameterizing notebooks, for example passing a date or environment name in from a job.<br><br>

<b>🎤 5. Where must a magic command appear in a cell?</b><br>
On the first line. Otherwise the cell runs in the notebook's default language.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A notebook's default language is Python. How do you run a SQL query in one of its cells?</b><br>
A. Wrap the query in <code>sql("...")</code> without <code>spark</code><br>
B. Start the cell with <code>%sql</code><br>
C. Change the notebook language for that run<br>
D. Use <code>%run sql</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> A magic command on the first line switches the language for that cell. <code>spark.sql("...")</code> in Python also works.</details><br><br>

<b>Q2. A team wants to reuse functions defined in <code>utils</code> inside <code>etl</code>, with access to the same variables. What should they use in <code>etl</code>?</b><br>
A. <code>dbutils.notebook.run("./utils", 60)</code><br>
B. <code>%run ./utils</code><br>
C. <code>import utils</code> (always works for notebooks)<br>
D. <code>%pip install ./utils</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>%run</code> executes the notebook in the same context. <code>dbutils.notebook.run</code> runs it separately, so its functions aren't available to the caller.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a DataFrame of 5 products with <code>id</code>, <code>name</code> and <code>price</code>. Register a temp view and query it with <code>%sql</code> for products above a given price</li><li>Add a widget <code>min_price</code> and use it to filter the DataFrame in Python</li><li>Move your DataFrame-creating code into a helper notebook and load it with <code>%run</code></li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. DataFrame and temp view (query it in a %sql cell:  SELECT * FROM products WHERE price > 100)
products = spark.createDataFrame(
    [(1, "Laptop", 899.0), (2, "Mouse", 25.0), (3, "Desk", 250.0), (4, "Lamp", 40.0), (5, "Chair", 150.0)],
    "id INT, name STRING, price DOUBLE",
)
products.createOrReplaceTempView("products")

# 2. Widget-driven filter
dbutils.widgets.text("min_price", "100", "Minimum price")
display(products.filter(products.price > float(dbutils.widgets.get("min_price"))))
dbutils.widgets.removeAll()

# 3. Put the 'products = ...' code in a notebook such as ./00_products_helper,
#    then run  %run ./00_products_helper  alone in a cell, and use 'products' here.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>A magic command must be the first line of a cell</li><li><code>display()</code> is interactive, <code>show()</code> is plain text, <code>print(df)</code> shows only the description</li><li>Temp views bridge Python and SQL</li><li>Widgets parameterize notebooks</li><li><code>%run</code> reuses code from another notebook in the same context</li></ul>
</div>

| Task | Code |
|---|---|
| SQL cell | `%sql` on the first line |
| Python → SQL | `df.createOrReplaceTempView("v")` |
| Text widget | `dbutils.widgets.text("name", "default", "Label")` |
| Read widget | `dbutils.widgets.get("name")` |
| Remove widgets | `dbutils.widgets.removeAll()` |
| Include notebook | `%run ./other_notebook` |

## Git commit

```
git add 00_platform_tour/00_02_notebook_basics.ipynb
git commit -m "restyle 00_02 notebook basics to lesson standard"
```